In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
df = pd.read_csv('marketing_AB.csv')
#df.head(5)

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


A/B Test

We conduct a one-sided A/B test to compare the conversion rates between the treatment group (ad) and the control group (psa).

H0: The conversion rates are the same
H1: The conversion rate for the ad group is higher

Suppose we take significant level alpha = 0.05, power beta = 0.1, 

In [5]:
from statsmodels.stats.proportion import proportions_ztest

# aggregate
summary = df.groupby('test group')['converted'].agg(
    users='count',
    conversions='sum',
    conversion_rate='mean'
)

print(summary)

             users  conversions  conversion_rate
test group                                      
ad          564577        14423         0.025547
psa          23524          420         0.017854


In [6]:
# make sure order is [ad, psa]
counts = summary.loc[['ad', 'psa'], 'conversions'].values
nobs = summary.loc[['ad', 'psa'], 'users'].values

# one-sided test (ad > psa)
z_stat, p_value = proportions_ztest(
    count=counts,
    nobs=nobs,
    alternative='larger'
)

print("z-stat:", z_stat)
print("p-value:", p_value)

z-stat: 7.3700781265454145
p-value: 8.526403580779863e-14


The test yields a z-statistic of 7.37 and a p-value of 8.5e−14. Since the p-value is far below the 0.05 significance level, we reject the null hypothesis. This provides strong statistical evidence that the ad treatment leads to a higher conversion rate.

Regression Analysis

To further understand the drivers of conversion, we perform a logistic regression using the features:

total ads (exposure intensity)

most ads day (day of peak exposure)

most ads hour (time of peak exposure)

The results indicate that both the intensity and timing of ad exposure significantly affect conversion. Higher ad exposure is associated with increased conversion odds, while exposure during certain days (e.g., early weekdays) and times (e.g., afternoon and evening) is more effective than late-night or early-morning exposure.

In [13]:
df_ad_group = df[df['test group'] == 'ad']

In [22]:
# --------- features ----------
X = df_ad_group[['total ads', 'most ads day', 'most ads hour']]
y = df_ad_group['converted']

# --------- column types ----------
num_cols = ['total ads']
cat_cols = ['most ads day', 'most ads hour']

# --------- preprocessing ----------
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop=[['Friday'], [0]]), cat_cols)
    ]
)

# --------- model ----------
model = LogisticRegression(max_iter=1000)

# --------- pipeline ----------
pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

# --------- split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --------- fit ----------
pipe.fit(X_train, y_train)

# --------- score ----------
print("Train score:", pipe.score(X_train, y_train))
print("Test score:", pipe.score(X_test, y_test))

Train score: 0.97332955468814
Test score: 0.9726788054837224


In [23]:
feature_names = (
    num_cols +
    list(pipe.named_steps['preprocess']
         .named_transformers_['cat']
         .get_feature_names_out(cat_cols))
)

coefs = pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs
}).sort_values(by='coef', ascending=False)

print(coef_df)

                   feature      coef
5     most ads day_Tuesday  0.528934
1      most ads day_Monday  0.508023
0                total ads  0.449273
6   most ads day_Wednesday  0.319366
3      most ads day_Sunday  0.307730
22        most ads hour_16  0.298055
20        most ads hour_14  0.163263
26        most ads hour_20  0.156628
27        most ads hour_21  0.138675
21        most ads hour_15  0.131329
4    most ads day_Thursday  0.122775
23        most ads hour_17  0.099473
28        most ads hour_22  0.098202
2    most ads day_Saturday  0.088061
29        most ads hour_23  0.069332
24        most ads hour_18  0.044360
25        most ads hour_19  0.033749
19        most ads hour_13 -0.031616
18        most ads hour_12 -0.127626
16        most ads hour_10 -0.182206
10         most ads hour_4 -0.188403
11         most ads hour_5 -0.203184
17        most ads hour_11 -0.210467
15         most ads hour_9 -0.224034
14         most ads hour_8 -0.342153
12         most ads hour_6 -0.345933
9

In [24]:
y_pred = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_pred)

np.float64(0.8102428896660834)

The coefficient for total ads is positive (0.449), indicating that higher ad exposure is associated with higher odds of conversion. Specifically, each additional ad increases the odds of conversion by approximately 57%, holding other factors constant.

The timing of ad exposure also plays an important role. Users whose peak exposure occurs between Sunday and Wednesday tend to have higher conversion odds, while exposure during midnight and early morning hours is associated with lower conversion odds.

All categorical effects are interpreted relative to the baseline categories (Friday and 12am). A positive coefficient indicates higher conversion odds compared to the baseline, while a negative coefficient indicates lower conversion odds.

To further investigate the treatment effect, I analyze whether the ad group contributes positively to conversion after controlling for exposure day and time.

In [20]:
# --------- features ----------
X_ctrl = df[['total ads', 'most ads day', 'most ads hour', 'test group']]
y_ctrl = df['converted']

# --------- column types ----------
num_cols = ['total ads']
cat_cols = ['most ads day', 'most ads hour', 'test group']

# --------- preprocessing ----------
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop=[['Friday'], [0], ['psa']]), cat_cols)
    ]
)

# --------- model ----------
model = LogisticRegression(max_iter=1000)

# --------- pipeline ----------
pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

# --------- split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_ctrl, y_ctrl, test_size=0.2, random_state=42
)

# --------- fit ----------
pipe.fit(X_train, y_train)

# --------- score ----------
print("Train score:", pipe.score(X_train, y_train))
print("Test score:", pipe.score(X_test, y_test))

Train score: 0.9735121577962932
Test score: 0.9734656226354137


In [25]:
feature_names = (
    num_cols +
    list(pipe.named_steps['preprocess']
         .named_transformers_['cat']
         .get_feature_names_out(cat_cols))
)

coefs = pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs
}).sort_values(by='coef', ascending=False)

print(coef_df)

                   feature      coef
5     most ads day_Tuesday  0.528934
1      most ads day_Monday  0.508023
0                total ads  0.449273
6   most ads day_Wednesday  0.319366
3      most ads day_Sunday  0.307730
22        most ads hour_16  0.298055
20        most ads hour_14  0.163263
26        most ads hour_20  0.156628
27        most ads hour_21  0.138675
21        most ads hour_15  0.131329
4    most ads day_Thursday  0.122775
23        most ads hour_17  0.099473
28        most ads hour_22  0.098202
2    most ads day_Saturday  0.088061
29        most ads hour_23  0.069332
24        most ads hour_18  0.044360
25        most ads hour_19  0.033749
19        most ads hour_13 -0.031616
18        most ads hour_12 -0.127626
16        most ads hour_10 -0.182206
10         most ads hour_4 -0.188403
11         most ads hour_5 -0.203184
17        most ads hour_11 -0.210467
15         most ads hour_9 -0.224034
14         most ads hour_8 -0.342153
12         most ads hour_6 -0.345933
9

After controlling for ad exposure intensity and timing, the coefficient for the ad group remains positive (0.366), indicating that the ad treatment still increases conversion odds relative to the PSA group. Specifically, the odds of conversion are approximately 44% higher for users in the ad group, holding other factors constant.

This suggests that the positive effect observed in the A/B test is not solely driven by differences in exposure patterns, but also reflects a genuine treatment effect.

In [28]:
summary = df.groupby('test group')['converted'].agg(['mean', 'count'])
p_ad = summary.loc['ad', 'mean']
p_psa = summary.loc['psa', 'mean']

abs_lift = p_ad - p_psa
rel_lift = (p_ad - p_psa) / p_psa

print(abs_lift, rel_lift)

0.007692453192201517 0.43085064022225833


The ad treatment increases conversion by 43% (relative lift), which translates to approximately 769 additional conversions per 10,000 users. 